In [1]:
import pandas as pd

import src.constants as C
from src.preprocessing import process_store_data
from src.features import attach_store_data, make_features, make_targets



# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always


In [2]:
store_df = pd.read_csv(C.STORE_FILE)
store_df = process_store_data(store_df)

df_train = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
df_train_store = attach_store_data(df_train, store_df)

# Shift Sales by 1 day per store to prevent any leakage of current-day sales into features
df_train_store = df_train_store.sort_values(['Store', 'Date'])
df_train_store['Sales_previous_day'] = df_train_store.groupby('Store')['Sales'].shift(1)

# TODO: Predict log-transformed sales?
df_features = make_features(df_train_store,
                            lags=C.LAGS,
                            roll_windows=C.ROLL_WINDOWS,
                            diffs=C.DIFFS)
targets = make_targets(df=df_train[['Date', 'Store', 'Sales']],
                       horizon=C.FORECAST_HORIZON)

#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)


C:\Users\Miltos.KALIKATZAR\AppData\Local\Temp\ipykernel_13076\2740746278.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)


In [4]:
pd.concat([
    df_features.dtypes,
    df_features.isna().sum()/len(df_features),
    df_features.nunique()
], axis=1).sort_values(1, ascending=False).round(2).to_csv('feature_summary.csv')

In [3]:
df_features.shape, targets.shape

((1017209, 79), (1017209, 42))